# BPE Unit Tutorial: From Information Compression to Tokenizers

This unit is organized around one central question:

> Is the BPE tokenizer used by large language models “understanding language,” or is it “compressing strings”?

We will begin with information compression, then gradually explain the BPE training process, how BPE encodes new sentences, how token boundaries are decided when prefixes overlap, and the fundamental difference between BPE and Huffman coding.

---

## Learning Objectives

After completing this unit, you should be able to answer:

1. Why do language models need tokenizers?
2. What exactly does BPE learn?
3. Why does BPE merge frequent character combinations into tokens?
4. When words share prefixes, such as `learner` and `learning`, how does BPE decide the segmentation?
5. Why is BPE not the same as Huffman coding?
6. Why is BPE a “trade-off between compression and compositionality”?

## 1. Starting from “Information Compression”

The basic idea of information compression is:

> Use fewer symbols to represent more information while preserving the truly important structure as much as possible.

For example:

```text
information compression
```

If represented character by character, it requires many characters:

```text
i / n / f / o / r / m / a / t / i / o / n / ... 
```

If represented word by word, it becomes:

```text
information / compression
```

This is obviously shorter.

However, direct word-level segmentation creates a problem: there are far too many words in the world. For example:

```text
learn
learns
learned
learning
learner
learnable
```

If every word is added to the vocabulary, the vocabulary becomes very large, and new words become difficult to handle.

Therefore, language models usually do not segment text directly into “characters” or “complete words.” Instead, they use an intermediate unit:

> **subword**

The goal of BPE is to automatically learn such subword units from a corpus.

## 2. Tokenizers Solve a “Representation Unit” Problem, Not a “Semantic Understanding” Problem

A language model cannot directly read natural language text. What it actually processes is a sequence of integers:

```text
text → tokenizer → token id sequence → Transformer
```

For example:

```text
information compression
```

may be segmented as:

```text
information / compression
```

and then mapped to:

```text
[3021, 9587]
```

The task of a tokenizer is not to understand the philosophical meaning of “information compression,” but to decide:

> Which discrete units should the original string be split into?

The core role of BPE is:

> To use corpus statistics to merge frequent adjacent characters or subwords into larger tokens.

## 3. A One-Sentence Definition of BPE

BPE stands for **Byte Pair Encoding**.

In modern NLP, BPE can usually be understood as:

> Starting from the smallest units, repeatedly count the most frequent adjacent symbol pairs in the corpus and merge them into new tokens.

What it learns is not a grammar rule or a semantic rule, but:

> An ordered table of string merge rules.

For example:

```text
l + e       → le
le + a      → lea
lea + r     → lear
lear + n    → learn
i + n       → in
in + g      → ing
learn + ing → learning
```

These rules are the core product of BPE training.

## 4. The BPE Training Process: From Characters to Subwords

Suppose the training corpus contains the following words and frequencies:

```text
low      5 times
lower    2 times
newest   6 times
widest   3 times
```

First, split each word into characters and add an end-of-word marker `</w>`:

```text
l o w </w>        × 5
l o w e r </w>    × 2
n e w e s t </w>  × 6
w i d e s t </w>  × 3
```

Then count the frequency of all adjacent pairs, such as:

```text
l o
o w
e s
s t
w e
...
```

If `e s` occurs most frequently, merge it:

```text
e + s → es
```

The corpus becomes:

```text
l o w </w>        × 5
l o w e r </w>    × 2
n e w es t </w>   × 6
w i d es t </w>   × 3
```

Then continue counting and merging, for example:

```text
es + t → est
l + o  → lo
lo + w → low
```

Eventually, we obtain a vocabulary and a sequence of merge rules.

## 5. What Is the Output of BPE Learning?

The learned result of BPE can be understood as three parts:

### 5.1 Initial Symbol Table

This may consist of characters or bytes.

For example:

```text
a, b, c, ..., z
中, 国, 人, ...
```

Or, in byte-level BPE, it consists of bytes from 0 to 255.

### 5.2 Merge Rules

This is the most important part. For example:

```text
l e        → le
le a       → lea
lea r      → lear
lear n     → learn
i n        → in
in g       → ing
learn ing  → learning
```

These rules are ordered, meaning they have a rank. Rules learned earlier have higher priority.

### 5.3 Token-to-ID Vocabulary Mapping

Finally, each token is assigned an integer id:

```text
learn    → 13521
ing      → 287
learning → 8163
```

What the model actually sees are these ids.

## 6. What Happens When BPE Encounters a New Sentence?

This is a key point in our discussion.

After BPE has learned from the training corpus, when it encounters a new sentence:

> It does not relearn rules, nor does it make temporary decisions based on semantics. Instead, it deterministically applies the learned merge rules.

For example, suppose the following rules have already been learned:

```text
l e        → le
le a       → lea
lea r      → lear
lear n     → learn
i n        → in
in g       → ing
learn ing  → learning
```

Now it encounters a new word:

```text
learning
```

First, split it into the smallest units:

```text
l e a r n i n g
```

Then merge according to the rules:

```text
l e a r n i n g
→ le a r n i n g
→ lea r n i n g
→ lear n i n g
→ learn i n g
→ learn in g
→ learn ing
→ learning
```

The final result is:

```text
learning
```

If the rule `learn + ing → learning` does not exist, the process stops at:

```text
learn / ing
```

Therefore, BPE encoding for a new sentence is:

```text
new sentence
↓
split into characters or bytes
↓
look up the learned merge rules
↓
repeatedly merge according to priority
↓
obtain the final token sequence
```

## 7. How Does BPE Decide Which Token a Repeated Prefix Belongs To?

This is an important detail.

Suppose the vocabulary contains:

```text
learn
learner
learning
```

Now the input is:

```text
learning
```

The question is: should BPE segment it as:

```text
learn / ing
```

or as:

```text
learning
```

The answer is not “by looking at semantics,” nor is it simply “longest match.” Instead:

> It depends on whether the merge rules contain a rule that further merges `learn` and `ing` into `learning`.

If such a rule exists:

```text
learn + ing → learning
```

then the final result is:

```text
learning
```

If not, it stops at:

```text
learn / ing
```

Thus, BPE token boundaries come from the priority of merge rules, not from linguistic judgments about roots and suffixes.

## 8. Why Might `learner` and `learning` Become Two Separate Tokens?

This is one of the important characteristics of BPE.

From a linguistic perspective, we might want:

```text
learner  → learn / er
learning → learn / ing
```

This would preserve the root and suffix structure.

However, BPE is not a morphological analyzer. It only looks at frequency.

If `learner` and `learning` are both very common in the corpus, BPE may very well learn:

```text
learner
learning
```

as two independent tokens.

This shows that the goal of BPE is not to discover the “most reasonable semantic units,” but to discover:

> Adjacent string fragments that are frequent in the training corpus and worth merging.

Therefore, BPE contains a fundamental tension:

```text
compression: frequent long fragments should be merged to reduce the number of tokens
compositionality: roots and suffixes should be preserved to support generalization
```

BPE makes a rough trade-off between the two: high-frequency fragments are merged, while low-frequency fragments remain split.

## 9. The Difference Between BPE and Huffman Coding

The question we started with was: will BPE converge toward an optimal compression method like Huffman coding?

The answer is: they are similar in some ways, but fundamentally different.

### What Does Huffman Coding Do?

Huffman coding assigns bit codes of different lengths to existing symbols:

```text
high-frequency symbol → short bit string
low-frequency symbol → long bit string
```

For example:

```text
the                  → 0
quantum field theory → 110101011...
```

It optimizes the average bit length.

### What Does BPE Do?

BPE does not assign bit codes of different lengths to tokens. Instead, it changes how the text is segmented:

```text
frequent adjacent fragment → merge into a new token
```

For example:

```text
l e a r n i n g
→ learn ing
→ learning
```

So:

```text
Huffman: optimizes code length
BPE: optimizes symbol segmentation
```

If we want true information-theoretic compression, the pipeline should be:

```text
text → tokenizer → token sequence → language model probability prediction → arithmetic coding / entropy coding
```

In that case, the whole system approaches the compression limit, not the tokenizer alone.

## 10. Chinese Characters, Latin Letters, and BPE

Different writing systems affect the initial units and merge paths of BPE.

For English with Latin letters, the path is usually:

```text
letters → subwords → words
```

For example:

```text
information
→ inform / ation
```

Chinese characters themselves are often already close to morpheme-level units:

```text
信息压缩
→ 信 / 息 / 压 / 缩
→ 信息 / 压缩
```

Therefore, Chinese may look naturally more “compact.” However, in computer encoding, one Chinese character usually occupies 3 bytes in UTF-8, while one ASCII letter usually occupies 1 byte.

BPE does not directly care whether “Chinese characters are more advanced” or whether “English is simpler.” What it cares about is:

> Whether certain adjacent units co-occur frequently in the corpus.

Therefore:

```text
信 + 息 → 信息
压 + 缩 → 压缩
```

If these pairs are frequent enough, they may be merged.

## 11. A Minimal BPE Implementation

Below, we implement a toy BPE algorithm with very little code to demonstrate its core mechanism.

Note: Real tokenizers are much more complex. For example, they may include Unicode normalization, byte-level processing, special tokens, whitespace handling, and other details. The code here only demonstrates the core idea of BPE.

In [1]:
from collections import Counter

def get_pair_stats(vocab):
    """
    vocab: dict[tuple[str], int]
    Example:
    {
        ('l','o','w','</w>'): 5,
        ('l','o','w','e','r','</w>'): 2
    }
    Return the weighted frequencies of all adjacent pairs.
    """
    pairs = Counter()
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs


def merge_pair(pair, vocab):
    """
    Merge the specified pair wherever it appears in vocab.
    For example, when pair=('l','o'):
    ('l','o','w') -> ('lo','w')
    """
    new_vocab = {}
    replacement = ''.join(pair)

    for symbols, freq in vocab.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(replacement)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_vocab[tuple(new_symbols)] = freq

    return new_vocab


def train_bpe(word_freqs, num_merges=10):
    """
    word_freqs: dict[str, int]
    num_merges: number of merge operations
    """
    vocab = {
        tuple(list(word) + ['</w>']): freq
        for word, freq in word_freqs.items()
    }

    merges = []

    for step in range(num_merges):
        pairs = get_pair_stats(vocab)
        if not pairs:
            break

        best_pair, best_freq = pairs.most_common(1)[0]
        merges.append((best_pair, best_freq))
        vocab = merge_pair(best_pair, vocab)

        print(f"Step {step + 1}: merge {best_pair} -> {''.join(best_pair)}  freq={best_freq}")

    return merges, vocab

In [2]:
word_freqs = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3,
}

merges, final_vocab = train_bpe(word_freqs, num_merges=10)

print("
Final vocab representation:")
for symbols, freq in final_vocab.items():
    print(symbols, "×", freq)

Step 1: merge ('e', 's') -> es  freq=9
Step 2: merge ('es', 't') -> est  freq=9
Step 3: merge ('est', '</w>') -> est</w>  freq=9
Step 4: merge ('l', 'o') -> lo  freq=7
Step 5: merge ('lo', 'w') -> low  freq=7
Step 6: merge ('n', 'e') -> ne  freq=6
Step 7: merge ('ne', 'w') -> new  freq=6
Step 8: merge ('new', 'est</w>') -> newest</w>  freq=6
Step 9: merge ('low', '</w>') -> low</w>  freq=5
Step 10: merge ('w', 'i') -> wi  freq=3

Final vocab representation:
('low</w>',) × 5
('low', 'e', 'r', '</w>') × 2
('newest</w>',) × 6
('wi', 'd', 'est</w>') × 3


When you run the code above, you will see that BPE selects the currently most frequent adjacent pair and merges it at each step.

This is what we mean by “greedy”: at each step, it performs the merge that looks best at the moment, but it does not guarantee a globally optimal solution.

## 12. Encoding New Words with the Learned Merge Rules

After training is complete, when facing a new word, we do not count the corpus again. Instead, we use the learned merge rules.

Below is a simplified BPE encoding function.

In [3]:
def encode_word(word, merges):
    """
    Encode a new word using the merges learned during training.
    Here, we repeatedly try merges in the order of the learned merge rules.
    """
    symbols = tuple(list(word) + ['</w>'])

    # pair -> rank
    merge_ranks = {pair: rank for rank, (pair, freq) in enumerate(merges)}

    while True:
        candidate_pairs = [
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        ]

        # Find pairs that exist in the current symbols and also appear in the merge rules.
        valid_pairs = [
            pair for pair in candidate_pairs
            if pair in merge_ranks
        ]

        if not valid_pairs:
            break

        # Select the pair with the earliest rank.
        best_pair = min(valid_pairs, key=lambda p: merge_ranks[p])
        symbols = tuple(merge_pair(best_pair, {symbols: 1}).keys())[0]

    # Remove the end-of-word marker for display only.
    return [s for s in symbols if s != '</w>']


for word in ["low", "lower", "lowest", "newer", "widest"]:
    print(word, "->", encode_word(word, merges))

low -> ['low</w>']
lower -> ['low', 'e', 'r']
lowest -> ['low', 'est</w>']
newer -> ['new', 'e', 'r']
widest -> ['wi', 'd', 'est</w>']


When observing the results, note the following:

1. Words that appeared in the training corpus may be merged more completely.
2. A new word does not necessarily become `<UNK>`; it can fall back to smaller subwords or characters.
3. If a certain merge path does not exist, BPE will not invent a long token out of nowhere.

## 13. Further Observation: `learner` and `learning`

Now construct a special corpus and see whether BPE splits `learner` and `learning` into `learn / er` and `learn / ing`, or directly merges them into complete tokens.

In [4]:
word_freqs2 = {
    "learn": 10,
    "learner": 8,
    "learning": 9,
    "learned": 4,
    "learnable": 2,
}

merges2, final_vocab2 = train_bpe(word_freqs2, num_merges=20)

print("
Encoding new words:")
for word in ["learn", "learner", "learning", "learners", "learnability"]:
    print(word, "->", encode_word(word, merges2))

Step 1: merge ('l', 'e') -> le  freq=35
Step 2: merge ('le', 'a') -> lea  freq=33
Step 3: merge ('lea', 'r') -> lear  freq=33
Step 4: merge ('lear', 'n') -> learn  freq=33
Step 5: merge ('learn', 'e') -> learne  freq=12
Step 6: merge ('learn', '</w>') -> learn</w>  freq=10
Step 7: merge ('learn', 'i') -> learni  freq=9
Step 8: merge ('learni', 'n') -> learnin  freq=9
Step 9: merge ('learnin', 'g') -> learning  freq=9
Step 10: merge ('learning', '</w>') -> learning</w>  freq=9
Step 11: merge ('learne', 'r') -> learner  freq=8
Step 12: merge ('learner', '</w>') -> learner</w>  freq=8
Step 13: merge ('learne', 'd') -> learned  freq=4
Step 14: merge ('learned', '</w>') -> learned</w>  freq=4
Step 15: merge ('learn', 'a') -> learna  freq=2
Step 16: merge ('learna', 'b') -> learnab  freq=2
Step 17: merge ('learnab', 'le') -> learnable  freq=2
Step 18: merge ('learnable', '</w>') -> learnable</w>  freq=2

Encoding new words:
learn -> ['learn</w>']
learner -> ['learner</w>']
learning -> ['lear

This experiment demonstrates a key point:

> BPE may first learn `learn`, but if `learner` and `learning` are frequent enough, it may continue merging `learn + er` and `learn + ing` into larger tokens.

Therefore, BPE does not guarantee that root and suffix structure will be preserved. Its output depends on corpus frequency and the order of the merge rules.

## 14. Summary of Common Misunderstandings

### Misunderstanding 1: BPE finds the most reasonable semantic units

Not necessarily.

BPE only looks at string co-occurrence frequency. It does not understand semantics, nor does it guarantee linguistically meaningful root or suffix boundaries.

---

### Misunderstanding 2: BPE uses longest match when prefixes overlap

The core of standard BPE is not simply longest match. It starts from the smallest units and gradually merges according to the rank of the merge rules.

---

### Misunderstanding 3: BPE is just Huffman coding

No.

Huffman coding assigns bit strings of different lengths to existing symbols; BPE changes the way symbols are segmented.

---

### Misunderstanding 4: Longer tokens are always better

Not necessarily.

Long tokens can reduce sequence length, but they may weaken compositionality, increase vocabulary pressure, and make the embeddings of rare tokens harder to learn well.

---

### Misunderstanding 5: Is BPE lossless or lossy?

As a tokenizer, BPE is usually lossless: the token sequence can be restored to the original string.

However, its representation of linguistic structure may be “biased”: it preserves high-frequency string structure, not necessarily semantic structure.

## 15. Final Takeaway

BPE can be understood as a statistical compression method designed for language models. Starting from characters or bytes, it repeatedly merges the most frequent adjacent symbols in the corpus, producing an ordered set of merge rules and a token vocabulary. When encountering a new sentence, BPE does not relearn, nor does it make semantic judgments. It deterministically segments the text according to the learned merge rules.

Therefore, what BPE learns is not roots, suffixes, or conceptual boundaries in the linguistic sense, but frequent string fragments in the training corpus that are worth merging. Words such as `learner` and `learning` may be split as `learn / er` and `learn / ing`, or they may each become complete tokens, depending on their corpus frequencies and the order of the merge rules.

This shows that BPE exists within a tension between compression and compositionality. If we pursue compression too aggressively, frequent long fragments will be merged as wholes, reducing the number of tokens but weakening internal structure. If we preserve compositionality too strongly, sequences become longer, increasing training and inference costs. The practical value of BPE lies precisely in this simple, greedy, and engineering-effective trade-off.

Compared with Huffman coding, BPE does not assign bit codes of different lengths to symbols. Instead, it changes the symbol segmentation of the text itself. What truly approaches information-theoretic optimal compression is not the tokenizer alone, but a full system consisting of the tokenizer, language-model probability prediction, and an entropy coder.

## 16. Exercises

### Exercise 1: Change the Number of Merges

Change `num_merges=20` above to:

```python
num_merges=5
num_merges=10
num_merges=30
```

Observe how the segmentations of `learning` and `learner` change.

Think about:

> Does a larger number of merges always produce longer tokens? Is that always good?

---

### Exercise 2: Change Corpus Frequencies

Increase the frequency of `learning` and decrease the frequency of `learner`.

Observe:

```python
"learning": 100
"learner": 1
```

Think about:

> Are high-frequency words more likely to be merged as whole tokens?

---

### Exercise 3: A Chinese Example

Following the code above, construct a Chinese corpus:

```python
{
    "信息": 10,
    "压缩": 10,
    "信息压缩": 8,
    "语言模型": 12,
    "大语言模型": 6
}
```

Observe whether BPE learns:

```text
信息
压缩
信息压缩
语言模型
大语言模型
```

Think about:

> Chinese characters themselves are close to morpheme-level units. How might this affect the BPE merge process?